# Hydro Temporal GNN — Kaggle launcher

Notebook ini hanya menemukan/menyiapkan project dan memanggil entry point modular. Seluruh preprocessing, graph construction, training, evaluasi, inference, dan submission tetap berada di `src/` dan `scripts/`.

In [ ]:
import platform
import sys
from pathlib import Path

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Working directory: {Path.cwd()}')
print(f'Kaggle working exists: {Path("/kaggle/working").exists()}')

In [ ]:
import shutil
import subprocess

WORKING_REPOSITORY = Path('/kaggle/working/graph-nn-tma-mdpl')
PROJECT_ROOT = WORKING_REPOSITORY / 'notebook-graph-modular'

if not PROJECT_ROOT.is_dir():
    uploaded_candidates = sorted(Path('/kaggle/input').glob('**/notebook-graph-modular'))
    if uploaded_candidates:
        source_project = uploaded_candidates[0]
        WORKING_REPOSITORY.mkdir(parents=True, exist_ok=True)
        shutil.copytree(
            source_project,
            PROJECT_ROOT,
            ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'outputs'),
        )
        print(f'Copied uploaded project: {source_project} -> {PROJECT_ROOT}')
    else:
        clone_command = [
            'git', 'clone', '--branch', 'modular-version',
            'https://github.com/ibrahim119920/graph-nn-tma-mdpl.git',
            str(WORKING_REPOSITORY),
        ]
        try:
            subprocess.run(clone_command, check=True)
        except subprocess.CalledProcessError as error:
            raise FileNotFoundError(
                'notebook-graph-modular tidak ditemukan. Upload project sebagai Kaggle Dataset '
                'atau aktifkan Internet lalu jalankan git clone branch modular-version. '
                f'Perintah gagal: {clone_command}'
            ) from error

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project root tidak tersedia setelah bootstrap: {PROJECT_ROOT}')

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'kaggle.json'
print(f'Project root: {PROJECT_ROOT}')
print(f'Config: {CONFIG_PATH}')

In [ ]:
DATASET_ROOT = Path('/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026')
SAMPLE_SUBMISSION = Path('/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026/sample_submission.csv')

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Kaggle dataset root tidak ditemukan: {DATASET_ROOT}')
if not SAMPLE_SUBMISSION.is_file():
    raise FileNotFoundError(f'sample_submission.csv tidak ditemukan: {SAMPLE_SUBMISSION}')
print(f'Dataset root OK: {DATASET_ROOT}')
print(f'Sample submission OK: {SAMPLE_SUBMISSION}')

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'smoke_test.py')],
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'run_pipeline.py'),
        '--config', str(CONFIG_PATH),
    ],
    check=True,
)

In [ ]:
import pandas as pd

OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
SUBMISSION_PATH = OUTPUT_ROOT / 'submissions' / 'submission.csv'
if not SUBMISSION_PATH.is_file():
    raise FileNotFoundError(f'Pipeline selesai tetapi submission tidak ditemukan: {SUBMISSION_PATH}')
submission = pd.read_csv(SUBMISSION_PATH)
print(f'Output root: {OUTPUT_ROOT}')
print(f'Submission: {SUBMISSION_PATH}')
print(f'Submission shape: {submission.shape}')
display(submission.head())

In [ ]:
%cd /kaggle/working/graph-nn-tma-mdpl/notebook-graph-modular

!zip -r /kaggle/working/codex-handoff.zip \
    outputs/experiments \
    outputs/logs \
    outputs/submissions